GPU RAM使用量：9.0 ~ 11.0 GB  
系統 RAM使用量：5.8 ~ 6.3 GB

In [1]:
!pip install transformers==4.39.3
!pip install -q accelerate bitsandbytes
!pip install -q FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fo

In [2]:
import json
from google.colab import drive

In [3]:
from datasets import Dataset

PDF 前處理

In [ ]:
# @title
!pip install pymupdf pandas

In [ ]:
# @title
import json
import re
from collections import defaultdict
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd


PDF_DIR = Path("./pdfs")
OUTPUT_DIR = Path("./outputs")
BY_COMPANY_DIR = OUTPUT_DIR / "by_company"
TABLES_PATH = OUTPUT_DIR / "parsed_policy_tables.json"
RAG_DIR = OUTPUT_DIR / "rag"
RAG_TEXT_JSONL_PATH = RAG_DIR / "policy_rag_text_chunks.jsonl"
RAG_TABLE_JSONL_PATH = RAG_DIR / "policy_rag_table_chunks.jsonl"
RAG_COMPANY_JSON_PATH = RAG_DIR / "policy_rag_company_index.json"
OUTPUT_DIR.mkdir(exist_ok=True)
BY_COMPANY_DIR.mkdir(exist_ok=True)
RAG_DIR.mkdir(exist_ok=True)

TABLE_HEADER_HINTS = (
    "項目", "項次", "給付", "比例", "等級", "保險金", "保險金額", "名稱",
    "方案", "期間", "天數", "限額", "單位", "職業", "年齡", "級距", "事故",
    "失能", "身故", "醫療", "住院", "門診", "海外", "突發疾病", "交通", "班機", "地區",
)
TABLE_KEYWORD_HINTS = TABLE_HEADER_HINTS + (
    "附表", "附錄", "保險期間", "保險責任", "不保事項", "海外突發疾病",
    "傷害醫療", "旅遊不便", "行程取消", "班機延誤", "醫療專機",
)


VERSION_SUFFIX_PATTERN = re.compile(r"_[0-9]{6,8}$")
PRODUCT_CODE_PATTERN = re.compile(r"商品代號[:：]\s*([A-Z0-9\-]+)")
ARTICLE_NO_PATTERN = re.compile(r"第\s*[一二三四五六七八九十百千零〇\d]+\s*條")
ARTICLE_START_PATTERN = re.compile(r"(?P<article>第\s*[一二三四五六七八九十百千零〇\d]+\s*條)")
BRACKET_TITLE_PATTERN = re.compile(r"【(?P<title>[^】\n]+)】")
CHAPTER_PATTERN = re.compile(r"第\s*[一二三四五六七八九十百千零〇\d]+\s*章\s*(?P<title>[^\n]*)")
COMPANY_PATTERN = re.compile(
    r"(?P<company>[\u4e00-\u9fffA-Za-z0-9]+?(?:人壽保險|人壽|產物保險|海上產險|產險|保險公司|保險))"
)
PLAN_NAME_TRIM_PATTERN = re.compile(r"(附約|附加條款|附加保險|主約)(?:\(.*?\))?$")
TABLE_LIKE_TITLE_PATTERN = re.compile(r"(附表|附錄|給付表|項目表|失能等級|給付比例|保險金額表|方案表)")
PAGE_NOISE_PATTERNS = [
    re.compile(r"^第\s*\d+\s*頁[，,]?\s*共\s*\d+\s*頁\s*$", re.MULTILINE),
    re.compile(r"^[A-Z0-9\-()（） ]+\s+\d+/\d+\s*$", re.MULTILINE),
]


def normalize_text(text):
    text = text.replace("\u3000", " ")
    text = text.replace("\uf9c1", "療")
    text = text.replace("\uf9f7", "立")
    text = text.replace("\uf941", "限")
    text = text.replace("\uf9fc", "識")
    text = text.replace("\uf9fa", "狀")
    text = text.replace("\uf965", "便")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def remove_page_noise(text):
    clean_text = text
    for pattern in PAGE_NOISE_PATTERNS:
        clean_text = pattern.sub("", clean_text)
    return normalize_text(clean_text)


def strip_version_suffix(name):
    return VERSION_SUFFIX_PATTERN.sub("", name)


def sanitize_filename(name):
    safe = re.sub(r"[^\w\-.]+", "_", name, flags=re.UNICODE).strip("._")
    return safe or "unknown"


def build_full_text_with_page_markers(pages):
    parts = []
    for page in pages:
        parts.append(f"[[PAGE:{page['page']}]]")
        parts.append(page["text"])
    return normalize_text("\n".join(parts))


def normalize_table_cell(cell):
    if cell is None:
        return ""
    return normalize_text(str(cell))


def pad_table_rows(rows):
    width = max((len(row) for row in rows), default=0)
    return [row + [""] * (width - len(row)) for row in rows], width


def looks_like_table_header(row):
    nonempty = [cell for cell in row if cell]
    if not nonempty:
        return False

    joined = " ".join(nonempty)
    keyword_hits = sum(1 for hint in TABLE_HEADER_HINTS if hint in joined)
    numeric_like = sum(1 for cell in nonempty if re.search(r"\d", cell))
    short_like = sum(1 for cell in nonempty if len(cell) <= 12)
    return keyword_hits > 0 or (len(nonempty) >= 2 and short_like >= len(nonempty) - 1 and numeric_like <= 1)


def combine_table_headers(header_rows, width):
    headers = []
    for column_index in range(width):
        parts = []
        for row in header_rows:
            cell = row[column_index].strip()
            if cell and cell not in parts:
                parts.append(cell)
        headers.append(" / ".join(parts) if parts else f"欄位{column_index + 1}")
    return headers


def flatten_table_data_rows(data_rows, column_headers):
    flattened_rows = []
    carry_forward = [""] * len(column_headers)

    for row in data_rows:
        resolved_row = list(row)
        for index, cell in enumerate(resolved_row):
            if cell:
                carry_forward[index] = cell
            elif carry_forward[index] and index < len(resolved_row) - 1:
                resolved_row[index] = carry_forward[index]

        pairs = []
        for index, cell in enumerate(resolved_row):
            if not cell:
                continue
            header = column_headers[index] if index < len(column_headers) else f"欄位{index + 1}"
            pairs.append(f"{header}: {cell}")

        if pairs:
            flattened_rows.append("；".join(pairs))

    return flattened_rows


def extract_table_keywords(record, header_rows, flattened_rows):
    keywords = []
    candidates = [
        record.get("company_name", ""),
        record.get("product_name", ""),
        record.get("plan_name", ""),
        record.get("document_type", ""),
        record.get("article_no", ""),
        record.get("article_title", ""),
        *(" ".join(row) for row in header_rows),
        *flattened_rows[:8],
    ]
    combined = "\n".join(text for text in candidates if text)

    for hint in TABLE_KEYWORD_HINTS:
        if hint in combined and hint not in keywords:
            keywords.append(hint)

    for value in (
        record.get("company_name", ""),
        record.get("product_name", ""),
        record.get("plan_name", ""),
        record.get("article_no", ""),
        record.get("article_title", ""),
    ):
        if value and value not in keywords:
            keywords.append(value)

    return keywords


def build_table_semantic_text(record, table):
    sections = [
        f"條文編號: {record.get('article_no', '')}",
        f"條文標題: {record.get('article_title', '')}",
        f"保險公司: {record.get('company_name', '')}",
        f"商品名稱: {record.get('product_name', '')}",
        f"方案名稱: {record.get('plan_name', '')}",
        f"文件類型: {record.get('document_type', '')}",
        f"表格頁碼: {table.get('page', '')}",
    ]
    if table.get("column_headers"):
        sections.append(f"表格欄位: {' | '.join(table['column_headers'])}")
    if table.get("keywords"):
        sections.append(f"表格關鍵字: {' | '.join(table['keywords'])}")
    if table.get("header_rows"):
        sections.append("表頭列:")
        sections.extend(" | ".join(row) for row in table["header_rows"])
    sections.append("表格內容:")
    sections.extend(table.get("flattened_rows") or [" | ".join(row) for row in table.get("rows", [])])
    return "\n".join(section for section in sections if section.strip())


def enrich_tables_for_records(records):
    header_cache = {}

    for record in records:
        enriched_tables = []

        for table in record.get("tables", []):
            raw_rows, width = pad_table_rows(table.get("rows", []))
            if not raw_rows:
                continue

            header_rows = []
            data_start_index = 0
            for row in raw_rows[:2]:
                if looks_like_table_header(row):
                    header_rows.append(row)
                    data_start_index += 1
                else:
                    break

            cache_key = (
                record.get("source_file", ""),
                record.get("article_no", ""),
                record.get("article_title", ""),
                width,
            )
            cached_header_rows = header_cache.get(cache_key, [])
            if not header_rows and cached_header_rows:
                header_rows = cached_header_rows
            elif header_rows:
                header_cache[cache_key] = header_rows

            column_headers = combine_table_headers(header_rows, width) if width else []
            data_rows = raw_rows[data_start_index:] if header_rows else raw_rows
            fallback_headers = column_headers or [f"欄位{i + 1}" for i in range(width)]
            flattened_rows = flatten_table_data_rows(data_rows, fallback_headers)
            keywords = extract_table_keywords(record, header_rows, flattened_rows)

            enriched_table = {
                **table,
                "header_rows": header_rows,
                "column_headers": column_headers,
                "flattened_rows": flattened_rows,
                "keywords": keywords,
                "cross_page_header_inferred": bool(not header_rows and cached_header_rows),
            }
            enriched_table["semantic_text"] = build_table_semantic_text(record, enriched_table)
            enriched_tables.append(enriched_table)

        record["tables"] = enriched_tables
        record["has_table"] = bool(enriched_tables)
        record["table_count"] = len(enriched_tables)

    return records


def extract_tables_from_page(page):
    page_tables = []

    try:
        finder = page.find_tables()
    except Exception:
        finder = None

    if not finder:
        return page_tables

    tables = getattr(finder, "tables", None) or []
    for table_index, table in enumerate(tables, start=1):
        rows = []
        try:
            extracted = table.extract()
        except Exception:
            extracted = []

        for row in extracted or []:
            normalized_row = [normalize_table_cell(cell) for cell in row]
            if any(cell for cell in normalized_row):
                rows.append(normalized_row)

        if not rows:
            continue

        page_tables.append({
            "table_index": table_index,
            "row_count": len(rows),
            "column_count": max(len(row) for row in rows) if rows else 0,
            "rows": rows,
        })

    return page_tables


def extract_text_from_pdf(pdf_path):
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            raw_text = page.get_text("text")
            pages.append({
                "page": page_index,
                "text": remove_page_noise(raw_text),
                "tables": extract_tables_from_page(page),
            })
    return pages


def detect_layout_profile(pdf_path, full_text):
    first_chunk = full_text[:4000]
    file_name = pdf_path.name

    if "商品代號" in first_chunk and "【" in first_chunk:
        return "fubon_bracketed"
    if "共同條款" in first_chunk and "第一條" in first_chunk:
        return "chaptered_policy"
    if "第一條" in first_chunk:
        return "article_plain"
    if "第 一 條" in first_chunk:
        return "article_spaced"
    if "保單條款" in file_name or "電子保單條款" in file_name:
        return "policy_booklet"
    return "generic"


def infer_product_name(pdf_path, full_text):
    stem_name = strip_version_suffix(pdf_path.stem)
    if "保險" in stem_name or "條款" in stem_name:
        return stem_name

    candidates = []
    for raw_line in full_text.splitlines()[:60]:
        line = normalize_text(raw_line)
        if not line:
            continue
        if "商品代號" in line:
            continue
        if "保險" not in line and "條款" not in line:
            continue
        if len(line) < 6:
            continue
        candidates.append(line)

    if candidates:
        candidates.sort(key=lambda line: ("第" in line and "條" in line, len(line)))
        return candidates[0]

    return stem_name


def infer_company_name(product_name, file_name, full_text):
    for source in (product_name, strip_version_suffix(Path(file_name).stem), full_text[:1500]):
        match = COMPANY_PATTERN.search(source)
        if match:
            return match.group("company").strip()
    return "未知保險公司"


def infer_document_type(product_name, file_name, full_text):
    joined = f"{file_name}\n{product_name}\n{full_text[:2000]}"
    if "附約" in joined or "附加條款" in joined or "附加保險" in joined:
        return "附約"
    if "主約" in joined:
        return "主約"
    if "本附約" in full_text[:3000]:
        return "附約"
    return "主約"


def infer_plan_name(product_name, company_name, document_type):
    plan_name = product_name
    if company_name and plan_name.startswith(company_name):
        plan_name = plan_name[len(company_name):].strip(" -_")

    if document_type in {"主約", "附約"}:
        plan_name = PLAN_NAME_TRIM_PATTERN.sub("", plan_name).strip()

    return plan_name or product_name


def detect_product_info(pdf_path, full_text, profile):
    product_code_match = PRODUCT_CODE_PATTERN.search(full_text)
    product_name = infer_product_name(pdf_path, full_text)
    company_name = infer_company_name(product_name, pdf_path.name, full_text)
    document_type = infer_document_type(product_name, pdf_path.name, full_text)
    plan_name = infer_plan_name(product_name, company_name, document_type)

    return {
        "layout_profile": profile,
        "product_code": product_code_match.group(1) if product_code_match else "",
        "product_name": product_name,
        "document_type": document_type,
        "company_name": company_name,
        "plan_name": plan_name,
    }


def extract_page_range(chunk_text):
    page_nums = sorted({int(page) for page in re.findall(r"\[\[PAGE:(\d+)\]\]", chunk_text)})
    return (
        min(page_nums) if page_nums else None,
        max(page_nums) if page_nums else None,
    )


def clean_chunk_content(chunk_text):
    clean_text = re.sub(r"\[\[PAGE:\d+\]\]", "", chunk_text)
    return remove_page_noise(clean_text)


def collect_tables_for_page_range(pages, page_start, page_end):
    tables = []
    if page_start is None or page_end is None:
        return tables

    for page in pages:
        page_no = page["page"]
        if page_start <= page_no <= page_end:
            for table in page.get("tables", []):
                tables.append({
                    "page": page_no,
                    "table_index": table["table_index"],
                    "row_count": table["row_count"],
                    "column_count": table["column_count"],
                    "rows": table["rows"],
                })
    return tables


def is_table_like_record(record):
    joined = f"{record.get('article_title', '')}\n{record.get('content', '')}"
    return record.get("has_table", False) or bool(TABLE_LIKE_TITLE_PATTERN.search(joined))


def infer_article_title(full_text_with_page, match):
    line_end = full_text_with_page.find("\n", match.end())
    if line_end == -1:
        line_end = len(full_text_with_page)
    inline_title = normalize_text(full_text_with_page[match.end():line_end]).strip(" ：:")
    if inline_title and len(inline_title) <= 80:
        return inline_title

    lookback_start = max(0, match.start() - 120)
    lookback_text = full_text_with_page[lookback_start:match.start()]
    bracket_matches = list(BRACKET_TITLE_PATTERN.finditer(lookback_text))
    if bracket_matches:
        return bracket_matches[-1].group("title").strip()

    chapter_matches = list(CHAPTER_PATTERN.finditer(lookback_text))
    if chapter_matches:
        chapter_title = chapter_matches[-1].group("title").strip()
        if chapter_title:
            return chapter_title

    return ""


def build_article_records(full_text_with_page, pages, product_info, source_file):
    matches = list(ARTICLE_START_PATTERN.finditer(full_text_with_page))
    records = []

    for index, match in enumerate(matches):
        start = match.start()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(full_text_with_page)
        chunk_text = full_text_with_page[start:end].strip()
        page_start, page_end = extract_page_range(chunk_text)
        clean_content = clean_chunk_content(chunk_text)
        tables = collect_tables_for_page_range(pages, page_start, page_end)
        article_no = re.sub(r"\s+", "", match.group("article"))
        article_title = infer_article_title(full_text_with_page, match)

        records.append({
            "source_file": source_file,
            "document_type": product_info["document_type"],
            "product_code": product_info["product_code"],
            "product_name": product_info["product_name"],
            "company_name": product_info["company_name"],
            "plan_name": product_info["plan_name"],
            "layout_profile": product_info["layout_profile"],
            "parse_status": "success",
            "article_no": article_no,
            "article_title": article_title,
            "page_start": page_start,
            "page_end": page_end,
            "has_table": bool(tables),
            "table_count": len(tables),
            "tables": tables,
            "content": clean_content,
        })

    return records


def build_fallback_record(full_text_with_page, pages, product_info, source_file):
    page_start, page_end = extract_page_range(full_text_with_page)
    tables = collect_tables_for_page_range(pages, page_start, page_end)
    return [{
        "source_file": source_file,
        "document_type": product_info["document_type"],
        "product_code": product_info["product_code"],
        "product_name": product_info["product_name"],
        "company_name": product_info["company_name"],
        "plan_name": product_info["plan_name"],
        "layout_profile": product_info["layout_profile"],
        "parse_status": "fallback",
        "article_no": "全文",
        "article_title": "全文內容",
        "page_start": page_start,
        "page_end": page_end,
        "has_table": bool(tables),
        "table_count": len(tables),
        "tables": tables,
        "content": clean_chunk_content(full_text_with_page),
    }]


def build_table_only_records(records):
    table_records = []

    for record in records:
        if not record.get("tables"):
            continue

        if not is_table_like_record(record):
            continue

        for table in record["tables"]:
            semantic_text = table.get("semantic_text") or build_table_semantic_text(record, table)
            table_records.append({
                "source_file": record["source_file"],
                "document_type": record["document_type"],
                "product_code": record["product_code"],
                "product_name": record["product_name"],
                "company_name": record["company_name"],
                "plan_name": record["plan_name"],
                "layout_profile": record["layout_profile"],
                "parse_status": "table_only",
                "article_no": f"{record['article_no']}_TABLE_{table['table_index']}",
                "article_title": record["article_title"] or f"表格 {table['table_index']}",
                "page_start": table["page"],
                "page_end": table["page"],
                "has_table": True,
                "table_count": 1,
                "tables": [table],
                "content": semantic_text,
                "chunk_kind": "table",
                "table_page": table["page"],
                "table_index": table["table_index"],
            })

    return table_records


def parse_articles(pages, product_info, source_file):
    full_text_with_page = build_full_text_with_page_markers(pages)
    records = build_article_records(full_text_with_page, pages, product_info, source_file)
    if records:
        records = enrich_tables_for_records(records)
        return records + build_table_only_records(records)
    fallback_records = build_fallback_record(full_text_with_page, pages, product_info, source_file)
    fallback_records = enrich_tables_for_records(fallback_records)
    return fallback_records + build_table_only_records(fallback_records)


def parse_pdf(pdf_path):
    pages = extract_text_from_pdf(pdf_path)
    full_text = normalize_text("\n".join(page["text"] for page in pages))
    profile = detect_layout_profile(pdf_path, full_text)
    product_info = detect_product_info(pdf_path, full_text, profile)
    records = parse_articles(pages, product_info, pdf_path.name)

    return {
        "pdf_path": pdf_path,
        "file_name": pdf_path.name,
        "record_count": len(records),
        "product_info": product_info,
        "records": records,
        "parse_status": "success" if all(
            record["parse_status"] in {"success", "table_only"} for record in records
        ) else "fallback",
    }


def build_batch_summary(parsed_files):
    file_count = len(parsed_files)
    file_names = [item["file_name"] for item in parsed_files]

    companies = sorted({
        item["product_info"]["company_name"]
        for item in parsed_files
        if item["product_info"]["company_name"]
    })
    product_names = sorted({
        item["product_info"]["product_name"]
        for item in parsed_files
        if item["product_info"]["product_name"]
    })
    plan_names = sorted({
        item["product_info"]["plan_name"]
        for item in parsed_files
        if item["product_info"]["plan_name"]
    })
    profiles = sorted({
        item["product_info"]["layout_profile"]
        for item in parsed_files
        if item["product_info"]["layout_profile"]
    })

    main_contracts = [
        item["product_info"]["product_name"]
        for item in parsed_files
        if item["product_info"]["document_type"] == "主約"
    ]
    riders = [
        item["product_info"]["product_name"]
        for item in parsed_files
        if item["product_info"]["document_type"] == "附約"
    ]
    fallback_files = [
        item["file_name"]
        for item in parsed_files
        if item["parse_status"] != "success"
    ]

    return {
        "input_mode": "single_file" if file_count == 1 else "multi_file",
        "file_count": file_count,
        "file_names": file_names,
        "companies": companies,
        "company_count": len(companies),
        "product_names": product_names,
        "product_count": len(product_names),
        "plan_names": plan_names,
        "plan_count": len(plan_names),
        "layout_profiles": profiles,
        "layout_profile_count": len(profiles),
        "has_main_contract": bool(main_contracts),
        "has_rider": bool(riders),
        "main_contract_count": len(main_contracts),
        "rider_count": len(riders),
        "main_contracts": sorted(main_contracts),
        "riders": sorted(riders),
        "has_multiple_companies": len(companies) > 1,
        "has_multiple_plans": len(plan_names) > 1,
        "fallback_file_count": len(fallback_files),
        "fallback_files": fallback_files,
    }


def attach_batch_metadata(parsed_files, batch_summary):
    company_to_main = {}
    for item in parsed_files:
        company_name = item["product_info"]["company_name"]
        if item["product_info"]["document_type"] == "主約":
            company_to_main.setdefault(company_name, []).append(item["product_info"]["product_name"])

    for item in parsed_files:
        product_info = item["product_info"]
        company_mains = company_to_main.get(product_info["company_name"], [])

        attached_to = ""
        if product_info["document_type"] == "附約" and len(company_mains) == 1:
            attached_to = company_mains[0]
        elif product_info["document_type"] == "主約":
            attached_to = product_info["product_name"]

        file_metadata = {
            "source_file_stem": strip_version_suffix(Path(item["file_name"]).stem),
            "input_mode": batch_summary["input_mode"],
            "file_count": batch_summary["file_count"],
            "has_main_contract": batch_summary["has_main_contract"],
            "has_rider": batch_summary["has_rider"],
            "main_contract_count": batch_summary["main_contract_count"],
            "rider_count": batch_summary["rider_count"],
            "company_count": batch_summary["company_count"],
            "plan_count": batch_summary["plan_count"],
            "layout_profile_count": batch_summary["layout_profile_count"],
            "has_multiple_companies": batch_summary["has_multiple_companies"],
            "has_multiple_plans": batch_summary["has_multiple_plans"],
            "attached_to": attached_to,
            "batch_file_names": " | ".join(batch_summary["file_names"]),
            "batch_companies": " | ".join(batch_summary["companies"]),
            "batch_plan_names": " | ".join(batch_summary["plan_names"]),
            "batch_main_contracts": " | ".join(batch_summary["main_contracts"]),
            "batch_riders": " | ".join(batch_summary["riders"]),
            "fallback_file_count": batch_summary["fallback_file_count"],
        }

        item["product_info"]["attached_to"] = attached_to
        item["file_metadata"] = file_metadata

        for record in item["records"]:
            record.update(file_metadata)
            record["attached_to"] = attached_to


def build_company_payload(company_name, company_records):
    source_files = sorted({record["source_file"] for record in company_records})
    products = sorted({record["product_name"] for record in company_records})
    plans = sorted({record["plan_name"] for record in company_records})
    main_contracts = sorted({
        record["product_name"]
        for record in company_records
        if record["document_type"] == "主約"
    })
    riders = sorted({
        record["product_name"]
        for record in company_records
        if record["document_type"] == "附約"
    })

    return {
        "company_name": company_name,
        "file_count": len(source_files),
        "source_files": source_files,
        "product_count": len(products),
        "products": products,
        "plan_count": len(plans),
        "plans": plans,
        "main_contracts": main_contracts,
        "riders": riders,
        "record_count": len(company_records),
        "records": company_records,
    }


def build_rag_text(record):
    sections = [
        f"保險公司: {record.get('company_name', '')}",
        f"商品名稱: {record.get('product_name', '')}",
        f"方案名稱: {record.get('plan_name', '')}",
        f"文件類型: {record.get('document_type', '')}",
        f"附加主約: {record.get('attached_to', '')}",
        f"條文: {record.get('article_no', '')} {record.get('article_title', '')}".strip(),
        f"內容: {record.get('content', '')}",
    ]
    return "\n".join(section for section in sections if section.strip())


def build_rag_table_text(record):
    table = record["tables"][0] if record.get("tables") else {}
    semantic_text = record.get("content") or table.get("semantic_text", "")
    sections = [
        f"保險公司: {record.get('company_name', '')}",
        f"商品名稱: {record.get('product_name', '')}",
        f"方案名稱: {record.get('plan_name', '')}",
        f"文件類型: {record.get('document_type', '')}",
        f"表格條文: {record.get('article_no', '')} {record.get('article_title', '')}".strip(),
        f"表格頁碼: {record.get('table_page', '')}",
        semantic_text,
    ]
    return "\n".join(section for section in sections if section.strip())


def write_rag_outputs(all_records):
    text_records = []
    table_records = []
    company_index = defaultdict(lambda: {
        "company_name": "",
        "products": set(),
        "plans": set(),
        "document_types": set(),
        "source_files": set(),
        "text_chunk_count": 0,
        "table_chunk_count": 0,
    })

    for record in all_records:
        company_name = record.get("company_name", "未知保險公司")
        company_bucket = company_index[company_name]
        company_bucket["company_name"] = company_name
        company_bucket["products"].add(record.get("product_name", ""))
        company_bucket["plans"].add(record.get("plan_name", ""))
        company_bucket["document_types"].add(record.get("document_type", ""))
        company_bucket["source_files"].add(record.get("source_file", ""))

        if record.get("chunk_kind") == "table":
            rag_record = {
                "id": f"{record['source_file']}::{record['article_no']}",
                "type": "table",
                "text": build_rag_table_text(record),
                "metadata": record,
            }
            table_records.append(rag_record)
            company_bucket["table_chunk_count"] += 1
        else:
            rag_record = {
                "id": f"{record['source_file']}::{record['article_no']}::{record.get('page_start')}",
                "type": "text",
                "text": build_rag_text(record),
                "metadata": record,
            }
            text_records.append(rag_record)
            company_bucket["text_chunk_count"] += 1

    with open(RAG_TEXT_JSONL_PATH, "w", encoding="utf-8") as file:
        for record in text_records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    with open(RAG_TABLE_JSONL_PATH, "w", encoding="utf-8") as file:
        for record in table_records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    company_payload = []
    for company_name, bucket in sorted(company_index.items()):
        company_payload.append({
            "company_name": company_name,
            "products": sorted(item for item in bucket["products"] if item),
            "plans": sorted(item for item in bucket["plans"] if item),
            "document_types": sorted(item for item in bucket["document_types"] if item),
            "source_files": sorted(item for item in bucket["source_files"] if item),
            "text_chunk_count": bucket["text_chunk_count"],
            "table_chunk_count": bucket["table_chunk_count"],
        })

    with open(RAG_COMPANY_JSON_PATH, "w", encoding="utf-8") as file:
        json.dump(company_payload, file, ensure_ascii=False, indent=2)

    return {
        "rag_text_jsonl": str(RAG_TEXT_JSONL_PATH),
        "rag_table_jsonl": str(RAG_TABLE_JSONL_PATH),
        "rag_company_json": str(RAG_COMPANY_JSON_PATH),
        "rag_text_chunk_count": len(text_records),
        "rag_table_chunk_count": len(table_records),
    }


def write_company_outputs(all_records):
    grouped_records = defaultdict(list)
    for record in all_records:
        grouped_records[record["company_name"]].append(record)

    company_paths = {}
    for company_name, company_records in grouped_records.items():
        payload = build_company_payload(company_name, company_records)
        output_path = BY_COMPANY_DIR / f"{sanitize_filename(company_name)}.json"
        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(payload, file, ensure_ascii=False, indent=2)
        company_paths[company_name] = str(output_path)

    return company_paths


def write_outputs(all_records, batch_summary):
    df = pd.DataFrame(all_records)

    csv_path = OUTPUT_DIR / "parsed_policy_chunks.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    jsonl_path = OUTPUT_DIR / "parsed_policy_chunks.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as file:
        for record in all_records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    table_records = [record for record in all_records if record.get("chunk_kind") == "table"]
    with open(TABLES_PATH, "w", encoding="utf-8") as file:
        json.dump(table_records, file, ensure_ascii=False, indent=2)

    company_paths = write_company_outputs(all_records)
    rag_outputs = write_rag_outputs(all_records)
    batch_summary["company_json_files"] = company_paths
    batch_summary["table_record_count"] = len(table_records)
    batch_summary["tables_json_file"] = str(TABLES_PATH)
    batch_summary.update(rag_outputs)

    summary_path = OUTPUT_DIR / "parsed_policy_summary.json"
    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(batch_summary, file, ensure_ascii=False, indent=2)

    print("\n解析完成")
    print(f"CSV 輸出：{csv_path}")
    print(f"JSONL 輸出：{jsonl_path}")
    print(f"表格 JSON 輸出：{TABLES_PATH}")
    print(f"RAG 文字 JSONL：{RAG_TEXT_JSONL_PATH}")
    print(f"RAG 表格 JSONL：{RAG_TABLE_JSONL_PATH}")
    print(f"RAG 公司索引：{RAG_COMPANY_JSON_PATH}")
    print(f"摘要輸出：{summary_path}")
    print(f"公司別 JSON 目錄：{BY_COMPANY_DIR}")


def main():
    pdf_files = sorted(PDF_DIR.glob("*.pdf"))

    if not pdf_files:
        print("找不到 PDF，請確認 ./pdfs 資料夾裡有 PDF 檔案")
        return

    parsed_files = []
    for pdf_path in pdf_files:
        print(f"正在解析：{pdf_path.name}")
        parsed = parse_pdf(pdf_path)
        print(
            f"解析出 {parsed['record_count']} 個 chunk，"
            f"profile={parsed['product_info']['layout_profile']}，"
            f"status={parsed['parse_status']}"
        )
        parsed_files.append(parsed)

    batch_summary = build_batch_summary(parsed_files)
    attach_batch_metadata(parsed_files, batch_summary)

    all_records = []
    for item in parsed_files:
        all_records.extend(item["records"])

    print(
        "批次判斷："
        f"{'單一檔案' if batch_summary['input_mode'] == 'single_file' else '多個檔案'}，"
        f"公司 {batch_summary['company_count']} 家，"
        f"方案 {batch_summary['plan_count']} 個，"
        f"主約 {batch_summary['main_contract_count']} 份，"
        f"附約 {batch_summary['rider_count']} 份，"
        f"fallback {batch_summary['fallback_file_count']} 份"
    )

    write_outputs(all_records, batch_summary)


if __name__ == "__main__":
    main()

可以在Colab的Secret裡設定HF_TOKEN

In [ ]:
# @title
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('hf_token'))

純提問版本，在 test_queries 裡輸入要問的問題

In [ ]:
# @title
import json
import re
import gc
import torch
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from FlagEmbedding import BGEM3FlagModel, FlagReranker

# ==========================================
# [Step 1] 意圖解析器 (Qwen2.5-Instruct 4-bit)
# ==========================================
class IntentParser:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        print("[Step 1] 載入意圖解析暨生成模型 (4-bit)...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype=torch.float16,
            low_cpu_mem_usage=True, quantization_config=quantization_config
        )

        self.system_prompt = """你是一個專業的保險查詢意圖解析器。
      請分析使用者的輸入，提取出保險公司、商品名稱、方案名稱、文件類型，並將剩餘問題濃縮為核心搜尋關鍵字。
      同時，判斷使用者的問題類型 (intent_type)：
      - text_qa：詢問條款定義、理賠條件等純文字概念。
      - table_qa：詢問費率、理賠比例、失能等級等需要查表的數值問題。
      - mixed：兩者皆有。
      你必須且只能輸出合法的 JSON 格式。
      範例：{"company_name": "富邦人壽", "intent_type": "table_qa", "search_query": "平安御守 第三級失能 給付比例"}"""

    def parse(self, user_query: str) -> dict:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_query}
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=True)
        response_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if not json_match: raise ValueError("JSON NotFound")
            parsed_data = json.loads(json_match.group())

            invalid_vals = ["未提及", "無", "不適用", "null", "none"]
            filters = {k: v for k, v in parsed_data.items()
                       if k not in ("search_query", "intent_type") and v and not any(i in str(v).lower() for i in invalid_vals)}

            return {
                "status": "success",
                "intent_type": parsed_data.get("intent_type", "mixed"),
                "filters": filters,
                "search_query": parsed_data.get("search_query", user_query) or user_query
            }
        except Exception as e:
            print(f"  [警告] 解析失敗: {e}")
            return {"status": "fallback", "intent_type": "mixed", "filters": {}, "search_query": user_query}

# ==========================================
# [Step 2 & 3] 混合檢索與父子映射器 (BGE-M3)
# ==========================================
class DatabaseRetriever:
    def __init__(self, text_jsonl_path, table_json_path, model_name="BAAI/bge-m3"):
        print("[Step 2] 載入雙軌資料庫與 BGE-M3...")
        self.model = BGEM3FlagModel(model_name, use_fp16=True)

        # --- 載入與向量化純文本 (Text Chunks) ---
        self.text_docs = []
        with open(text_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                if doc.get("chunk_kind") != "table":
                    self.text_docs.append(doc)

        print(f"  計算 {len(self.text_docs)} 筆純文字 Embeddings...")
        text_inputs = [f"【{d['article_no']} {d['article_title']}】\n{d['content']}" for d in self.text_docs]
        enc_text = self.model.encode(text_inputs, return_dense=True, return_sparse=True)
        self.text_dense = enc_text["dense_vecs"]
        self.text_sparse = enc_text["lexical_weights"]

        # --- 載入與向量化表格 (Table Chunks) ---
        with open(table_json_path, "r", encoding="utf-8") as f:
            self.table_docs = json.load(f)

        print(f"  計算 {len(self.table_docs)} 筆表格 Embeddings...")
        table_inputs = [f"【{d['article_no']} {d['article_title']}】\n{d.get('semantic_text', '')}\n關鍵字:{d.get('keywords', '')}"
                        for d in self.table_docs]
        if table_inputs:
            enc_table = self.model.encode(table_inputs, return_dense=True, return_sparse=True)
            self.table_dense = enc_table["dense_vecs"]
            self.table_sparse = enc_table["lexical_weights"]
        else:
            self.table_dense, self.table_sparse = [], []

        self.file_groups = defaultdict(list)
        self.global_to_local = {}
        for idx, doc in enumerate(self.text_docs):
            fname = doc.get("source_file", "unknown")
            self.global_to_local[idx] = {"file_name": fname, "local_idx": len(self.file_groups[fname])}
            self.file_groups[fname].append(doc)

    def _filter(self, docs, filters):
        matched = []
        for idx, doc in enumerate(docs):
            if all(str(v).lower() in str(doc.get(k, "")).lower() for k, v in filters.items()):
                matched.append(idx)
        return matched if matched else list(range(len(docs)))

    def _calculate_rrf(self, query_dense, query_sparse, candidates, target_dense, target_sparse, top_k):
        if not candidates: return []

        candidate_dense = target_dense[candidates]
        dense_scores = (candidate_dense @ query_dense.T).squeeze().tolist()
        if type(dense_scores) is not list: dense_scores = [dense_scores]

        sparse_scores = [self.model.compute_lexical_matching_score(query_sparse, target_sparse[idx]) for idx in candidates]

        dense_sorted = [candidates[i] for i, _ in sorted(enumerate(dense_scores), key=lambda x: x[1], reverse=True)]
        sparse_sorted = [candidates[i] for i, _ in sorted(enumerate(sparse_scores), key=lambda x: x[1], reverse=True)]

        rrf = defaultdict(float)
        for r, idx in enumerate(dense_sorted, 1): rrf[idx] += 1.0 / (60 + r)
        for r, idx in enumerate(sparse_sorted, 1): rrf[idx] += 1.0 / (60 + r)

        return sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def _compress_table(self, query_dense, table_doc, top_k_rows=5):
        flattened = table_doc.get("flattened_rows", [])
        headers = table_doc.get("header_rows", [])

        if not flattened or len(flattened) <= top_k_rows:
            return "\n".join(headers + flattened) if headers else str(table_doc.get("content", ""))

        row_enc = self.model.encode(flattened, return_dense=True)["dense_vecs"]
        scores = (row_enc @ query_dense.T).squeeze()
        if scores.ndim == 0: scores = [scores.item()]
        else: scores = scores.tolist()

        scored_rows = sorted(zip(scores, flattened), key=lambda x: x[0], reverse=True)
        top_rows = [r[1] for r in scored_rows[:top_k_rows]]

        return "\n".join(headers + top_rows) + "\n... [資料過長已截斷]"

    def hybrid_search_and_prepare(self, intent_type: str, query: str, filters: dict, top_k=10):
        q_enc = self.model.encode([query], return_dense=True, return_sparse=True)
        q_dense, q_sparse = q_enc["dense_vecs"][0], q_enc["lexical_weights"][0]

        final_contexts = []

        if intent_type in ("text_qa", "mixed"):
            text_cands = self._filter(self.text_docs, filters)
            text_results = self._calculate_rrf(q_dense, q_sparse, text_cands, self.text_dense, self.text_sparse, top_k)

            intervals_by_file = defaultdict(list)
            for idx, _ in text_results:
                mapping = self.global_to_local[idx]
                fname, l_idx = mapping["file_name"], mapping["local_idx"]
                max_idx = len(self.file_groups[fname]) - 1
                intervals_by_file[fname].append([max(0, l_idx - 1), min(max_idx, l_idx + 1)])

            for fname, intervals in intervals_by_file.items():
                intervals.sort()
                merged = [intervals[0]]
                for curr in intervals[1:]:
                    if curr[0] <= merged[-1][1] + 1: merged[-1][1] = max(merged[-1][1], curr[1])
                    else: merged.append(curr)

                for start, end in merged:
                    docs = self.file_groups[fname][start:end+1]
                    final_contexts.append({
                        "source_type": "text",
                        "company_name": docs[0].get("company_name"),
                        "product_name": docs[0].get("product_name"),
                        "included_articles": [d['article_no'] for d in docs],
                        "expanded_content": "\n".join([f"【{d['article_no']} {d['article_title']}】\n{d['content']}" for d in docs])
                    })

        if intent_type in ("table_qa", "mixed") and len(self.table_dense) > 0:
            table_cands = self._filter(self.table_docs, filters)
            table_results = self._calculate_rrf(q_dense, q_sparse, table_cands, self.table_dense, self.table_sparse, top_k)

            for idx, _ in table_results:
                doc = self.table_docs[idx]
                compressed_table = self._compress_table(q_dense, doc, top_k_rows=8)
                final_contexts.append({
                    "source_type": "table",
                    "company_name": doc.get("company_name"),
                    "product_name": doc.get("product_name"),
                    "included_articles": [doc.get('article_no')],
                    "expanded_content": f"【{doc['article_no']} {doc['article_title']} (部分節錄)】\n{compressed_table}"
                })

        return final_contexts

# ==========================================
# [Step 4] 原生交叉重排器
# ==========================================
class ContextReranker:
    def __init__(self, model_name="BAAI/bge-reranker-v2-m3"):
        print("[Step 4] 載入原生 Reranker 模型...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        self.model.eval()

    def rerank(self, query: str, contexts: list, top_k=3, threshold=0.05):
        if not contexts: return []

        pairs = [[query, ctx["expanded_content"]] for ctx in contexts]
        scores = []
        with torch.no_grad():
            for q, d in pairs:
                inputs = self.tokenizer(
                    q, d, padding=True, truncation=True, max_length=512, return_tensors="pt"
                ).to(self.model.device)
                outputs = self.model(**inputs)
                logit = outputs.logits[0][0].item()
                score = 1 / (1 + torch.exp(torch.tensor(-logit)).item())
                scores.append(score)

        for idx, ctx in enumerate(contexts):
            ctx["score"] = scores[idx]

        contexts.sort(key=lambda x: x["score"], reverse=True)
        return [ctx for ctx in contexts[:top_k] if ctx["score"] >= threshold]

# ==========================================
# 系統整合與執行測試
# ==========================================
def main():
    # 1. 系統初始化
    print("=== 系統初始化開始 ===")
    parser = IntentParser(model_id="Qwen/Qwen2.5-7B-Instruct")
    db = DatabaseRetriever(
        text_jsonl_path="/content/parsed_policy_chunks.jsonl",
        table_json_path="/content/parsed_policy_tables.json"
    )
    reranker = ContextReranker()
    print("=== 系統初始化完成 ===\n")

    # 您的自訂生成 System Prompt
    generation_system_prompt = """你是一個專業且嚴謹的保險理賠文件分析系統。
你的唯一任務是根據使用者提供的【參考資料】，精準回答使用者的保險問題。

# 核心處理原則（請嚴格遵守）：
1. 資訊核實：僅基於【參考資料】回答。若提供的資料與問題無關，或無法推導出答案，請直接回答：「很抱歉，目前的參考資料中未包含足夠資訊來解答此問題。」絕對禁止編造或推測。
2. 過濾雜訊：檢索的文本可能包含不相關的條文（例如除外責任、其他給付項目）。請先判斷文本是否真正符合使用者的問題情境，過濾掉無關的段落，切勿照抄無關內容。
3. 表格數據提煉：若參考資料包含「表格數據」，請針對使用者的條件（如：特定失能等級、特定手術）精確檢索對應的數值或比例。**絕對禁止直接列印或複製貼上原始的表格字串或序列資料**，必須將其轉換為自然語言說明。
4. 簡明扼要：直接回答核心問題，並在說明邏輯時保持簡潔。禁止無意義的段落重複。

# 規定輸出格式：
請一律使用以下格式輸出你的分析結果（若無相關資訊，請略過說明與來源，僅輸出結論）：
【結論】：（用一句話直接回答使用者的問題，包含關鍵金額或比例）
【說明】：（條理分明地列出理賠條件或計算方式，可使用條列式）
【資料來源】：（標明答案出處，例如：某某保險-某某附約-第X條）
"""

    # 2. 測試查詢
    test_queries = [
        "如果我在國外發生嚴重意外需要醫療專機協助，這項專機服務有涵蓋哪些特定的國家或地區嗎？",
        "如果被保險人不幸因意外事故身故，家屬（受益人）向保險公司申請理賠時，必須準備哪些文件？",
        "條款中對於「意外傷害事故」的明確定義是什麼？"
    ]

    # ==========================================


    for query in test_queries:
        print(f"\n使用者提問: {query}")
        print("-" * 60)

        # Step 1: 意圖解析與路由判定
        intent = parser.parse(query)
        intent_type = intent.get('intent_type', 'mixed')
        filters = intent.get('filters', {})
        search_query = intent.get('search_query', query)

        print(f"[Step 1] 查詢路由: {intent_type}")
        print(f"         解析條件: {filters}")
        print(f"         搜尋字串: {search_query}")

        # Step 2 & 3: 雙軌檢索與動態上下文組裝
        expanded_contexts = db.hybrid_search_and_prepare(
            intent_type=intent_type, query=search_query, filters=filters, top_k=10
        )
        print(f"[Step 2 & 3] 雙軌檢索與動態壓縮完成，產出 {len(expanded_contexts)} 個父文本區段")

        # Step 4: 交叉重排
        final_contexts = reranker.rerank(search_query, expanded_contexts, top_k=2)
        current_query_contexts = [ctx.get('expanded_content', '') for ctx in final_contexts]
        print(f"[Step 4] 重排完成，保留 {len(final_contexts)} 個高分文本")

        for rank, ctx in enumerate(final_contexts, 1):
            source_label = "表格" if ctx.get('source_type') == "table" else "純文本"
            print(f"  ➤ 推薦排名 {rank} (分數: {ctx.get('score', 0):.4f}) [{source_label}]")

        # ==========================================
        # [Step 5] 複用千問模型進行答案生成 (LLM Generator)
        # ==========================================
        print("[Step 5] 複用 Qwen2.5 進行最終答案生成...")

        # 建立注入的 Context 文本
        # 建立注入的 Context 文本
        formatted_contexts = []
        for idx, ctx in enumerate(final_contexts, 1):
            source_label = "表格數據" if ctx.get('source_type') == "table" else "條文本文"
            formatted_contexts.append(
                f"[文獻 {idx}]\n"
                f"來源：{ctx.get('company_name', '未知')} - {ctx.get('product_name', '未知')}\n"
                f"類型：{source_label}\n"
                f"內容：\n{ctx.get('expanded_content', '')}\n"
            )

        context_str = "\n\n".join(formatted_contexts) if formatted_contexts else "無檢索到相關資料。"
        max_context_chars = 3500  # 約對應 1500~2000 tokens，依 VRAM 彈性調整
        if len(context_str) > max_context_chars:
            context_str = context_str[:max_context_chars] + "\n\n...[因長度限制，後續文獻已省略]..."

        # 使用 XML 標籤隔離上下文與問題
        user_content = (
            f"<context>\n"
            f"{context_str}\n"
            f"</context>\n\n"
            f"<query>\n"
            f"{query}\n"
            f"</query>\n\n"
            f"請根據 <context> 內的資訊，嚴格依照【結論】、【說明】、【資料來源】的格式回答 <query> 中的問題。"
        )

        messages = [
            {"role": "system", "content": generation_system_prompt},
            {"role": "user", "content": user_content}
        ]

        gen_text = parser.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # 修正點 A：加入 truncation 與 max_length 限制，防止 Attention 矩陣記憶體爆炸
        # 建議設為 3072 或 4096，保留約 1024 Token 給輸出
        gen_inputs = parser.tokenizer(
            [gen_text],
            return_tensors="pt",
            truncation=False
        ).to(parser.model.device)

        # 修正點 B：在呼叫 generate 之前，強制執行垃圾回收與清空 CUDA Cache
        gc.collect()
        torch.cuda.empty_cache()

        # 執行推理
        with torch.no_grad():
            gen_outputs = parser.model.generate(
                **gen_inputs,
                max_new_tokens=1024,
                temperature=0.01,
                do_sample=False,
                repetition_penalty=1.1,  # 新增此參數：大於 1.0 即可有效懲罰重複生成的 Token (建議值 1.05 ~ 1.2)
                no_repeat_ngram_size=5   # 新增此參數：禁止連續 5 個 token 的片語重複出現
            )

        # 解碼輸出
        answer = parser.tokenizer.decode(
            gen_outputs[0][gen_inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        print("\n💡 模型最終回答：")
        print(answer)
        print("=" * 60)
        # 修正點 C：單輪對話結束後再次清理顯存，避免碎片化 (Fragmentation) 累積
        del gen_inputs
        del gen_outputs
        gc.collect()
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

含輸出RAGAS評估用的格式 (執行時間約為15~21分鐘)

In [4]:
# @title
import json
import re
import gc
import torch
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from FlagEmbedding import BGEM3FlagModel, FlagReranker

# ==========================================
# [Step 1] 意圖解析器 (Qwen2.5-Instruct 4-bit)
# ==========================================
class IntentParser:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        print("[Step 1] 載入意圖解析暨生成模型 (4-bit)...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype=torch.float16,
            low_cpu_mem_usage=True, quantization_config=quantization_config
        )

        self.system_prompt = """你是一個專業的保險查詢意圖解析器。
      請分析使用者的輸入，提取出保險公司、商品名稱、方案名稱、文件類型，並將剩餘問題濃縮為核心搜尋關鍵字。
      同時，判斷使用者的問題類型 (intent_type)：
      - text_qa：詢問條款定義、理賠條件等純文字概念。
      - table_qa：詢問費率、理賠比例、失能等級等需要查表的數值問題。
      - mixed：兩者皆有。
      你必須且只能輸出合法的 JSON 格式。
      範例：{"company_name": "富邦人壽", "intent_type": "table_qa", "search_query": "平安御守 第三級失能 給付比例"}"""

    def parse(self, user_query: str) -> dict:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_query}
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=True)
        response_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if not json_match: raise ValueError("JSON NotFound")
            parsed_data = json.loads(json_match.group())

            invalid_vals = ["未提及", "無", "不適用", "null", "none"]
            filters = {k: v for k, v in parsed_data.items()
                       if k not in ("search_query", "intent_type") and v and not any(i in str(v).lower() for i in invalid_vals)}

            return {
                "status": "success",
                "intent_type": parsed_data.get("intent_type", "mixed"),
                "filters": filters,
                "search_query": parsed_data.get("search_query", user_query) or user_query
            }
        except Exception as e:
            print(f"  [警告] 解析失敗: {e}")
            return {"status": "fallback", "intent_type": "mixed", "filters": {}, "search_query": user_query}

# ==========================================
# [Step 2 & 3] 混合檢索與父子映射器 (BGE-M3)
# ==========================================
class DatabaseRetriever:
    def __init__(self, text_jsonl_path, table_json_path, model_name="BAAI/bge-m3"):
        print("[Step 2] 載入雙軌資料庫與 BGE-M3...")
        self.model = BGEM3FlagModel(model_name, use_fp16=True)

        # --- 載入與向量化純文本 (Text Chunks) ---
        self.text_docs = []
        with open(text_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                if doc.get("chunk_kind") != "table":
                    self.text_docs.append(doc)

        print(f"  計算 {len(self.text_docs)} 筆純文字 Embeddings...")
        text_inputs = [f"【{d['article_no']} {d['article_title']}】\n{d['content']}" for d in self.text_docs]
        enc_text = self.model.encode(text_inputs, return_dense=True, return_sparse=True)
        self.text_dense = enc_text["dense_vecs"]
        self.text_sparse = enc_text["lexical_weights"]

        # --- 載入與向量化表格 (Table Chunks) ---
        with open(table_json_path, "r", encoding="utf-8") as f:
            self.table_docs = json.load(f)

        print(f"  計算 {len(self.table_docs)} 筆表格 Embeddings...")
        table_inputs = [f"【{d['article_no']} {d['article_title']}】\n{d.get('semantic_text', '')}\n關鍵字:{d.get('keywords', '')}"
                        for d in self.table_docs]
        if table_inputs:
            enc_table = self.model.encode(table_inputs, return_dense=True, return_sparse=True)
            self.table_dense = enc_table["dense_vecs"]
            self.table_sparse = enc_table["lexical_weights"]
        else:
            self.table_dense, self.table_sparse = [], []

        self.file_groups = defaultdict(list)
        self.global_to_local = {}
        for idx, doc in enumerate(self.text_docs):
            fname = doc.get("source_file", "unknown")
            self.global_to_local[idx] = {"file_name": fname, "local_idx": len(self.file_groups[fname])}
            self.file_groups[fname].append(doc)

    def _filter(self, docs, filters):
        matched = []
        for idx, doc in enumerate(docs):
            if all(str(v).lower() in str(doc.get(k, "")).lower() for k, v in filters.items()):
                matched.append(idx)
        return matched if matched else list(range(len(docs)))

    def _calculate_rrf(self, query_dense, query_sparse, candidates, target_dense, target_sparse, top_k):
        if not candidates: return []

        candidate_dense = target_dense[candidates]
        dense_scores = (candidate_dense @ query_dense.T).squeeze().tolist()
        if type(dense_scores) is not list: dense_scores = [dense_scores]

        sparse_scores = [self.model.compute_lexical_matching_score(query_sparse, target_sparse[idx]) for idx in candidates]

        dense_sorted = [candidates[i] for i, _ in sorted(enumerate(dense_scores), key=lambda x: x[1], reverse=True)]
        sparse_sorted = [candidates[i] for i, _ in sorted(enumerate(sparse_scores), key=lambda x: x[1], reverse=True)]

        rrf = defaultdict(float)
        for r, idx in enumerate(dense_sorted, 1): rrf[idx] += 1.0 / (60 + r)
        for r, idx in enumerate(sparse_sorted, 1): rrf[idx] += 1.0 / (60 + r)

        return sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]

    def _compress_table(self, query_dense, table_doc, top_k_rows=5):
        flattened = table_doc.get("flattened_rows", [])
        headers = table_doc.get("header_rows", [])

        if not flattened or len(flattened) <= top_k_rows:
            return "\n".join(headers + flattened) if headers else str(table_doc.get("content", ""))

        row_enc = self.model.encode(flattened, return_dense=True)["dense_vecs"]
        scores = (row_enc @ query_dense.T).squeeze()
        if scores.ndim == 0: scores = [scores.item()]
        else: scores = scores.tolist()

        scored_rows = sorted(zip(scores, flattened), key=lambda x: x[0], reverse=True)
        top_rows = [r[1] for r in scored_rows[:top_k_rows]]

        return "\n".join(headers + top_rows) + "\n... [資料過長已截斷]"

    def hybrid_search_and_prepare(self, intent_type: str, query: str, filters: dict, top_k=10):
        q_enc = self.model.encode([query], return_dense=True, return_sparse=True)
        q_dense, q_sparse = q_enc["dense_vecs"][0], q_enc["lexical_weights"][0]

        final_contexts = []

        if intent_type in ("text_qa", "mixed"):
            text_cands = self._filter(self.text_docs, filters)
            text_results = self._calculate_rrf(q_dense, q_sparse, text_cands, self.text_dense, self.text_sparse, top_k)

            intervals_by_file = defaultdict(list)
            for idx, _ in text_results:
                mapping = self.global_to_local[idx]
                fname, l_idx = mapping["file_name"], mapping["local_idx"]
                max_idx = len(self.file_groups[fname]) - 1
                intervals_by_file[fname].append([max(0, l_idx - 1), min(max_idx, l_idx + 1)])

            for fname, intervals in intervals_by_file.items():
                intervals.sort()
                merged = [intervals[0]]
                for curr in intervals[1:]:
                    if curr[0] <= merged[-1][1] + 1: merged[-1][1] = max(merged[-1][1], curr[1])
                    else: merged.append(curr)

                for start, end in merged:
                    docs = self.file_groups[fname][start:end+1]
                    final_contexts.append({
                        "source_type": "text",
                        "company_name": docs[0].get("company_name"),
                        "product_name": docs[0].get("product_name"),
                        "included_articles": [d['article_no'] for d in docs],
                        "expanded_content": "\n".join([f"【{d['article_no']} {d['article_title']}】\n{d['content']}" for d in docs])
                    })

        if intent_type in ("table_qa", "mixed") and len(self.table_dense) > 0:
            table_cands = self._filter(self.table_docs, filters)
            table_results = self._calculate_rrf(q_dense, q_sparse, table_cands, self.table_dense, self.table_sparse, top_k)

            for idx, _ in table_results:
                doc = self.table_docs[idx]
                compressed_table = self._compress_table(q_dense, doc, top_k_rows=8)
                final_contexts.append({
                    "source_type": "table",
                    "company_name": doc.get("company_name"),
                    "product_name": doc.get("product_name"),
                    "included_articles": [doc.get('article_no')],
                    "expanded_content": f"【{doc['article_no']} {doc['article_title']} (部分節錄)】\n{compressed_table}"
                })

        return final_contexts

# ==========================================
# [Step 4] 原生交叉重排器
# ==========================================
class ContextReranker:
    def __init__(self, model_name="BAAI/bge-reranker-v2-m3"):
        print("[Step 4] 載入原生 Reranker 模型...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        self.model.eval()

    def rerank(self, query: str, contexts: list, top_k=3, threshold=0.05):
        if not contexts: return []

        pairs = [[query, ctx["expanded_content"]] for ctx in contexts]
        scores = []
        with torch.no_grad():
            for q, d in pairs:
                inputs = self.tokenizer(
                    q, d, padding=True, truncation=True, max_length=512, return_tensors="pt"
                ).to(self.model.device)
                outputs = self.model(**inputs)
                logit = outputs.logits[0][0].item()
                score = 1 / (1 + torch.exp(torch.tensor(-logit)).item())
                scores.append(score)

        for idx, ctx in enumerate(contexts):
            ctx["score"] = scores[idx]

        contexts.sort(key=lambda x: x["score"], reverse=True)
        return [ctx for ctx in contexts[:top_k] if ctx["score"] >= threshold]

# ==========================================
# 系統整合與執行測試
# ==========================================
def main():
    # 1. 系統初始化
    print("=== 系統初始化開始 ===")
    parser = IntentParser(model_id="Qwen/Qwen2.5-7B-Instruct")
    db = DatabaseRetriever(
        text_jsonl_path="/content/parsed_policy_chunks.jsonl",
        table_json_path="/content/parsed_policy_tables.json"
    )
    reranker = ContextReranker()
    print("=== 系統初始化完成 ===\n")

    # 您的自訂生成 System Prompt
    generation_system_prompt = """你是一個專業且嚴謹的保險理賠文件分析系統。
你的唯一任務是根據使用者提供的【參考資料】，精準回答使用者的保險問題。

# 核心處理原則（請嚴格遵守）：
1. 資訊核實：僅基於【參考資料】回答。若提供的資料與問題無關，或無法推導出答案，請直接回答：「很抱歉，目前的參考資料中未包含足夠資訊來解答此問題。」絕對禁止編造或推測。
2. 過濾雜訊：檢索的文本可能包含不相關的條文（例如除外責任、其他給付項目）。請先判斷文本是否真正符合使用者的問題情境，過濾掉無關的段落，切勿照抄無關內容。
3. 表格數據提煉：若參考資料包含「表格數據」，請針對使用者的條件（如：特定失能等級、特定手術）精確檢索對應的數值或比例。**絕對禁止直接列印或複製貼上原始的表格字串或序列資料**，必須將其轉換為自然語言說明。
4. 簡明扼要：直接回答核心問題，並在說明邏輯時保持簡潔。禁止無意義的段落重複。

# 規定輸出格式：
請一律使用以下格式輸出你的分析結果（若無相關資訊，請略過說明與來源，僅輸出結論）：
【結論】：（用一句話直接回答使用者的問題，包含關鍵金額或比例）
【說明】：（條理分明地列出理賠條件或計算方式，可使用條列式）
【資料來源】：（標明答案出處，例如：某某保險-某某附約-第X條）
"""

    # 2. 測試查詢
    test_queries = [
        "如果我在國外發生嚴重意外需要醫療專機協助，這項專機服務有涵蓋哪些特定的國家或地區嗎？",
        "如果被保險人不幸因意外事故身故，家屬（受益人）向保險公司申請理賠時，必須準備哪些文件？",
        "條款中對於「意外傷害事故」的明確定義是什麼？",
        "如果保險公司沒有依約定提供醫療專機服務，請問被保險人最高可以拿到多少補償金？",
        "我搭乘的客機因為天候因素延遲了 10 小時才抵達目的地，導致我下機時保險期間已經結束了。如果我在這延遲的 10 小時內發生意外，並在事故發生後的第 200 天身故，這樣還可以理賠意外身故保險金嗎？需要符合什麼條件？",

        "我在日本滑雪受重傷住院，當地醫生說情況危急需要回國治療。但我等不及保險公司安排，自己先聯絡了私人醫療專機回台灣。請問保險公司還會負擔這筆我自行安排的專機費用嗎？如果不會，保險公司會怎麼處理我的保單？",
        "我幫我 12 歲的小孩投保，如果他不幸在旅行中發生意外身故，理賠金的項目會和一般成年人一樣叫做「意外身故保險金」嗎？額度上有沒有什麼特別的法規限制？",
        "被保險人原本就因為生病導致左眼失明，這次在保險有效期間內，不幸因為車禍意外導致右眼也失明。請問這樣「意外失能保險金」會怎麼理賠？會賠全盲（100%）的額度嗎？",
        "我特地去韓國的知名診所進行醫美整形手術，結果手術失敗引發嚴重併發症，當地醫院說有生命危險。這種緊急情況下，我可以申請你們的醫療專機送我回國嗎？",
        "條款上寫說「故意行為」不賠。那如果是幫我買這份保險的人（要保人）為了詐保，故意開車撞斷我的一條腿（導致我失能），身為被保險人的我，還能向保險公司申請這筆失能保險金嗎？",

        "如果我在國外旅遊時，突然心肌梗塞發作，導致我整個人昏倒，頭部重摔在柏油路上造成嚴重腦震盪。這個頭部的傷害可以申請「意外傷害」理賠嗎？",
        "如果在出國期間，我的托運行李被航空公司弄丟了，或者班機無故取消導致我必須多花錢住一晚飯店，這份保險有提供班機延誤或行李遺失的定額補償嗎？",
        "如果在國外租車自駕，我不小心撞到路人導致對方骨折住院，這份合約裡面的責任險會幫我賠償對方的醫藥費跟精神撫慰金嗎？",
        "這次出國我會帶我的黃金獵犬一起去，請問這幾份保單有包含寵物如果在國外生病的醫療保險，或是寵物咬傷別人的責任險嗎？",
        "申請海外醫療專機時，打給急難專線必須口頭告知哪些特定的個人資料與當地醫院資訊？",

        "辦理意外身故理賠時，條款有規定保險公司有權利對屍體進行相驗，或指定醫生幫被保險人體檢嗎？",
        "投保後在契約還沒生效前書面終止，保費會全額退嗎？若期間過了一半才終止，未滿期保費怎麼算？",
        "被保險人遭遇意外身故，但當初填要保書時沒填受益人，理賠金會給誰？是變遺產還是有優先繼承人？",
        "兩個月前曾因心臟病住院，下個月去越南旅遊時心臟病復發需要專機，這份附約會理賠嗎？有既往症限制嗎？",
        "海外車禍導致右手大拇指與食指從關節處完全斷裂，意外失能保險金是賠全額還是按部位比例？比例是多少？",

        "符合專機條件且已申請，但保險公司因故無法提供，家屬自行花百萬包機回台，保險公司是賠460萬還是全額核銷？",
        "保單10月10日中午12點到期。飛機11點30分降落，被保險人12點05分在機場門口被車撞，這樣能理賠嗎？",
        "投保前右眼已失明。本次旅遊因意外撞擊導致左眼也失明，醫生判定雙眼全盲。失能保險金會扣除右眼部分嗎？",
        "14 歲青少年在國外感染急性腦炎在醫院身故，屬於未成年，家屬可以申請「喪葬費用保險金」嗎？",
        "受傷後眼睛並非完全沒光感，但日常生活已無法自理且無法矯正，條款中「雙目失明」的醫學審定標準為何？",

        "在巴黎地鐵隨身行李、護照與手機被扒竊，有當地警察局報案證明，這份契約的財物損失補償限額是多少？",
        "國外跌倒骨折就醫花費數萬元，未達失能也沒有叫專機，要填哪份理賠申請書來報銷海外實支實付醫療費？",
        "出發前一天確診傳染病被強制隔離，導致機票飯店行程全部取消，這份保險有提供行程取消的損失補償嗎？",
        "搭乘的飛機在海外失蹤找不到遺體，受益人必須等民法宣告死亡多年才能申請身故金嗎？契約有特別時效嗎？",
        "在新安東京海上產險的海外緊急援助服務中，被保險人如果需要在事故當地安排海外喪葬事宜，其費用給付責任的上限是多少元？"
    ]
    # === 新增：準備 Ragas 評估用的資料容器 ===
    ragas_questions = []
    ragas_contexts = []
    ragas_answers = []

    # 填入您已有的 ground_truth (必須與 test_queries 長度、順序一致)
    ragas_ground_truths = [
        "根據醫療專機運送附約條款，「服務區域」涵蓋的特定國家或地區包含：中國大陸（含香港、澳門地區）、日本、南韓、越南、新加坡、菲律賓、印尼、馬來西亞、緬甸、泰國、寮國、柬埔寨、帛琉及馬爾地夫",
        "受益人申領「意外身故保險金或喪葬費用保險金」時，必須檢具下列五項文件：保險金申請書。保險單或其謄本（或其他投保證明文件。相驗屍體證明書或死亡診斷書（必要時保險公司得要求提供意外傷害事故證明文件）。被保險人除戶戶籍謄本。受益人的身分證明",
        "條款中對於「意外傷害事故」的明確定義為：「指非由疾病引起之外來突發事故」",
        "如果被保險人符合醫療專機運送條件，但保險公司未依約提供服務，或提供的服務規格不符約定（除可歸責於被保險人或其代理人的情形外），保險公司應給付補償金，該補償金最高為新臺幣四百六十萬元",
        "可以理賠。 首先，被保險人以乘客身分搭乘領有載客執照之交通工具，因故延遲抵達而非被保險人所能控制者，保單會自動延長有效期限至您終止乘客身分為止（最長延長不超過二十四小時），因此延遲的 10 小時內仍屬於保單有效期間。 其次，一般規定需在事故發生後 180 日內身故才理賠，但若超過 180 日死亡，只要受益人能證明被保險人之死亡與該次意外傷害事故「具有因果關係」，就不受 180 日的限制，依然可以申請意外身故保險金理賠",

        "保險公司不會負擔您自行安排的專機費用，因為條款明定保險公司不負擔被保險人或其親屬未經保險公司同意而自行處理所生的額外費用。 如果您已符合海外醫療專機運送服務的條件，卻另外接受非保險公司安排的專機運送服務，保險公司的處理方式是：不再提供海外醫療專機運送服務，而是改按「已繳保險費」並加計自保單生效日起至保險期間屆滿日止「按年利率 5% 計算之利息」的總金額，將此費用退還給您。",
        "理賠金的項目名稱不會叫做「意外身故保險金」，而是會變更為「喪葬費用保險金」。 在額度上有特別的法規限制：未滿 15 足歲的被保險人，其所投保的喪葬費用保險金額總和（不限單一保險公司）「不得超過遺產及贈與稅法第十七條有關遺產稅喪葬費扣除額之半數」。超過該限額的部分，保險公司不負給付責任，並會無息退還超過部分的已繳保險費。",
        "保險公司不會直接理賠全盲（100%）的額度。 條款規定，被保險人因本次意外事故所致的失能，如合併以前（含本契約訂立前）的失能，雖然可適用較嚴重項目（雙目失明）的意外失能保險金，但以前的失能（左眼失明）會被「視同已給付意外失能保險金，並予以扣除」。因此，理賠金額將是全盲額度扣除單眼失明額度後的差額。",
        "不可以申請。 根據海外醫療專機運送附約的「除外責任」條款，若被保險人是「以中華民國境外醫療為目的而進行之器官移植、醫學美容或其他治療行為」，保險公司不負提供海外醫療專機運送服務的責任。因此，特地前往海外進行醫美手術所引發的緊急狀況無法申請專機服務。",
        "可以申請。 雖然條款中將「要保人、被保險人的故意行為」列為除外責任，保險公司通常不負給付責任，但條款也明確設有例外規定：因「要保人」故意行為（除被保險人自己的故意行為外）導致被保險人失能時，保險公司「仍給付保險金」。因此，如果是要保人的故意行為導致您失能，身為被保險人的您依然可以申請失能保險金。",

        "條款中對於「意外傷害事故」的明確定義為：「指非由疾病引起之外來突發事故」。由於您的頭部傷害（腦震盪）是起因於心肌梗塞（疾病）發作導致昏倒所造成，這屬於由疾病所引發的事故，不符合「非由疾病引起」的意外傷害定義，因此通常無法申請意外傷害相關的理賠。",
        "視您投保的具體方案而定，部分方案有提供定額給付，部分則為實支實付。 例如，在「平安御守」方案中，針對班機延誤提供的是**「班機延誤保險(定額給付-累進式)」，而行李遺失則提供「行李損失保險(定額給付)」的定額補償。但若是投保「平安福」方案，其班機延誤與行李損失保險則是採取「實支實付」**的方式理賠，針對班機延誤實際額外支出的膳食、住宿及交通費用，或行李遺失的實際價值進行賠付，而非定額補償。",
        "不會賠償。 根據「個人賠償責任保險」的特別不保事項條款明定，被保險人因所有、使用或管理之飛機、船舶及依當地法令定義之「車輛」所致之賠償責任，保險公司不予賠償。因此，租車自駕所引發的交通事故責任，屬於除外不保範圍。",
        "在行李損失保險的特別不保事項中，也明確將「各種動物或植物」排除在外。 至於寵物咬傷別人的責任險，保單並未直接列出「寵物責任」，但「個人賠償責任保險」主要承保被保險人對於第三人體傷或財物受損依法應負之賠償責任。因為條款中並未將「寵物造成的損害」列為除外責任，若您在法律上須為您的寵物咬傷人負擔損害賠償責任，理論上可依個人賠償責任保險提出理賠申請，由保險公司依實際狀況審核賠付。",
        "申請時，您或親屬（代理人）必須以電話向急難專線告知以下四項資訊：被保險人之全名、身分證字號、護照號碼、出國日期(目的)、保單號碼、出生日期。救助機構可與被保險人、其親屬或其代理人聯絡之電話號碼被保險人住院之當地醫院電話號碼及地址（或主治醫師姓名、聯絡方式）。簡要描述住院發生之地點、被保險人狀況及所需之救助。 若未告知上述前三款事項，保險公司有權拒絕提供海外醫療專機運送服務。",

        "辦理身故理賠時，受益人僅需提供「相驗屍體證明書或死亡診斷書」（必要時提供意外傷害事故證明）即可。 保險公司「得對被保險人的身體予以檢驗，另得徵詢其他醫師之醫學專業意見」的權利，僅規範在申領「意外失能保險金」或「傷害醫療保險金」時適用，並不適用於身故理賠。",
        "保單條款約定，若要保人終止契約，保險公司會按保險契約「實際有效之天數」或「已經過期間」計算應繳保費，並返還已繳保費與應繳保費間的差額（即未滿期保費退還）。 因此，若在契約完全未生效前終止，由於經過期間為零，保費應會全額退還。若期間過了一半才終止，保險公司會扣除已過半期間的對應保費，將剩餘另一半未到期的保費（未滿期保險費）退還給您。",
        "被保險人遭遇意外身故，但當初填要保書時沒填受益人，理賠金會給誰？是變遺產還是有優先繼承人？ 如果當初沒有指定受益人，理賠金會給「被保險人之法定繼承人」。 條款明定，若身故保險金尚未給付或受益人先於被保險人身故且未另行指定時，以法定繼承人為受益人，且法定繼承人之順序及應得保險金之比例，會直接「適用民法繼承編相關規定」來分配。",
        "不會理賠，因為有既往症的明確限制。 「海外醫療專機運送附約」的除外責任規定：「被保險人在本附約生效前一百八十日以內曾接受診療之疾病」，保險公司不負提供醫療專機服務的責任。「海外突發疾病」的定義也明確限制必須是生效前九十日或一百八十日以內，未曾接受該疾病之診療者。 由於您兩個月前（約 60 天）才因心臟病住院治療，落在條款限制的 90 日或 180 日內，因此屬於既往症，無法啟動突發疾病理賠與專機服務。",
        "意外失能保險金不會賠全額，而是會「按部位比例」來計算給付金額。 根據保單的「失能程度與保險金給付表」，「一手拇指及食指缺失者」屬於第 8 級失能，其給付比例為保險金額的 30%。保險公司會以您投保的意外失能保險金額乘上 30% 來給付。",

        "保險公司會理賠最高 460 萬元的補償金，不會全額核銷家屬自行包機的百萬費用。 條款明定，被保險人若符合醫療專機運送條件，但保險公司未依約提供服務，保險公司應給付新臺幣四百六十萬元的補償金。同時，為達成海外醫療專機運送所生之相關交通工具費用，不包括因被保險人、其親屬或其代理人未經本公司同意，自行處理所生的額外費用。",
        "不能理賠。 雖然保單規定搭乘領有載客執照之交通工具因故延遲可自動延長保險期間，但最多僅延長至「被保險人終止乘客身分時為止」。飛機已於 11 點 30 分降落，被保險人下機抵達機場門口時已非客機乘客，因此自動延長效力已經終止。事故發生在 12 點 05 分，已經超過保單 12 點的到期時間，故無法理賠。",
        "會扣除。 條款中「意外失能保險金的給付」明確約定，被保險人因本次意外事故所致之失能，如合併以前（含本契約訂立前）的失能，可按較嚴重的項目（雙目失明）給付失能保險金；但是以前的失能（右眼失明），視同已給付意外失能保險金，保險公司在理賠時應予以扣除",
        "不可以申請「喪葬費用保險金」。 在條款中，「喪葬費用保險金」實為未滿15足歲被保險人的「意外身故保險金」變更而來，其給付前提必須是遭遇「非由疾病引起之外來突發事故」（即意外傷害事故）致死。感染急性腦炎屬於疾病，不符合意外傷害的定義，因此無法申請此項理賠。（註：若因突發疾病身故，僅能透過海外緊急救援服務條款申請當地的喪葬費用或遺體運送費用補償）。",
        "條款對於「失明」的醫學審定標準為：視力永久在萬國式視力表 0.02 以下。 此外，失明的定義也包含眼球喪失、摘出，以及僅能辨明暗，或辨眼前一公尺以內手動，或辨眼前五公分以內指數者。因此，即使仍保有微弱光感，只要符合上述標準之一，即可判定為雙目失明。",

        "這三項物品的理賠狀況與限額各有不同：隨身行李（包包等一般物品）：適用「行李損失保險」，理賠包含竊盜。對於每件物品之損失，最高以新臺幣 8,000 元為限，總理賠金額以保單載明之保險金額為上限。護照（旅行文件）：適用「旅行文件損失保險」，保險公司會賠償重新申辦護照的行政規費（依您投保的具體方案，採實支實付或定額給付，最高以保險金額為限）。手機（行動電話）：不予理賠。條款在行李損失保險中明確將「行動電話」列為特別不保事項物品；另外若有附加「行動電話強盜搶奪補償保險」，該條款僅承保遭「強盜或搶奪」的狀況，無法證明為強盜或搶奪的扒竊（竊盜）會被視為遺失，亦不在理賠範圍內。",
        "您需要填寫「保險金申請書」（或稱理賠申請書），來申領「傷害醫療保險金（實支實付型）」。 跌倒骨折屬於意外傷害事故，辦理理賠時除了申請書外，還需同時檢附保險單或謄本、醫療診斷書、醫療費用明細或收據，以及受益人身分證明等文件。",
        "根據「旅程取消保險」條款，承保的取消原因中，與被保險人身體健康相關的條件僅限於被保險人、配偶或三親等內親屬「死亡或病危」。一般確診傳染病遭到強制隔離，若未達「病危」程度，並不屬於約定的理賠範圍。此外，特別不保事項也明定，若於保險契約生效前因傳染病防治法須接受強制檢疫、隔離處置或罹患法定傳染病，保險公司不負理賠責任。不需要等民法宣告死亡多年。 根據條款中的「失蹤處理」約定，被保險人因約定的意外傷害事故失蹤，只要於戶籍資料所載失蹤之日起滿一年仍未尋獲，或者受益人能提出證明文件足以認為被保險人極可能因該次意外事故而死亡者，保險公司就會先行給付意外身故保險金。 關於契約的特別時效，條款中明定：由保險契約所生的權利，自得為請求之日起，經過兩年不行使而消滅。",
        "根據條款中的「失蹤處理」約定，被保險人因約定的意外傷害事故失蹤，只要於戶籍資料所載失蹤之日起滿一年仍未尋獲，或者受益人能提出證明文件足以認為被保險人極可能因該次意外事故而死亡者，保險公司就會先行給付意外身故保險金。 關於契約的特別時效，條款中明定：由保險契約所生的權利，自得為請求之日起，經過兩年不行使而消滅。",
        "在事故當地（需在海外）安排喪葬事宜所需的費用，給付上限為新臺幣五萬元。"
    ]
    # ==========================================


    for query in test_queries:
        print(f"\n使用者提問: {query}")
        print("-" * 60)

        # Step 1: 意圖解析與路由判定
        intent = parser.parse(query)
        intent_type = intent.get('intent_type', 'mixed')
        filters = intent.get('filters', {})
        search_query = intent.get('search_query', query)

        print(f"[Step 1] 查詢路由: {intent_type}")
        print(f"         解析條件: {filters}")
        print(f"         搜尋字串: {search_query}")

        # Step 2 & 3: 雙軌檢索與動態上下文組裝
        expanded_contexts = db.hybrid_search_and_prepare(
            intent_type=intent_type, query=search_query, filters=filters, top_k=10
        )
        print(f"[Step 2 & 3] 雙軌檢索與動態壓縮完成，產出 {len(expanded_contexts)} 個父文本區段")

        # Step 4: 交叉重排
        final_contexts = reranker.rerank(search_query, expanded_contexts, top_k=2)
        current_query_contexts = [ctx.get('expanded_content', '') for ctx in final_contexts]
        print(f"[Step 4] 重排完成，保留 {len(final_contexts)} 個高分文本")

        for rank, ctx in enumerate(final_contexts, 1):
            source_label = "表格" if ctx.get('source_type') == "table" else "純文本"
            print(f"  ➤ 推薦排名 {rank} (分數: {ctx.get('score', 0):.4f}) [{source_label}]")

        # ==========================================
        # [Step 5] 複用千問模型進行答案生成 (LLM Generator)
        # ==========================================
        print("[Step 5] 複用 Qwen2.5 進行最終答案生成...")

        # 建立注入的 Context 文本
        # 建立注入的 Context 文本
        formatted_contexts = []
        for idx, ctx in enumerate(final_contexts, 1):
            source_label = "表格數據" if ctx.get('source_type') == "table" else "條文本文"
            formatted_contexts.append(
                f"[文獻 {idx}]\n"
                f"來源：{ctx.get('company_name', '未知')} - {ctx.get('product_name', '未知')}\n"
                f"類型：{source_label}\n"
                f"內容：\n{ctx.get('expanded_content', '')}\n"
            )

        context_str = "\n\n".join(formatted_contexts) if formatted_contexts else "無檢索到相關資料。"
        max_context_chars = 3500  # 約對應 1500~2000 tokens，依 VRAM 彈性調整
        if len(context_str) > max_context_chars:
            context_str = context_str[:max_context_chars] + "\n\n...[因長度限制，後續文獻已省略]..."

        # 使用 XML 標籤隔離上下文與問題
        user_content = (
            f"<context>\n"
            f"{context_str}\n"
            f"</context>\n\n"
            f"<query>\n"
            f"{query}\n"
            f"</query>\n\n"
            f"請根據 <context> 內的資訊，嚴格依照【結論】、【說明】、【資料來源】的格式回答 <query> 中的問題。"
        )

        messages = [
            {"role": "system", "content": generation_system_prompt},
            {"role": "user", "content": user_content}
        ]

        gen_text = parser.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # 修正點 A：加入 truncation 與 max_length 限制，防止 Attention 矩陣記憶體爆炸
        # 建議設為 3072 或 4096，保留約 1024 Token 給輸出
        gen_inputs = parser.tokenizer(
            [gen_text],
            return_tensors="pt",
            truncation=False
        ).to(parser.model.device)

        # 修正點 B：在呼叫 generate 之前，強制執行垃圾回收與清空 CUDA Cache
        gc.collect()
        torch.cuda.empty_cache()

        # 執行推理
        with torch.no_grad():
            gen_outputs = parser.model.generate(
                **gen_inputs,
                max_new_tokens=1024,
                temperature=0.01,
                do_sample=False,
                repetition_penalty=1.1,  # 新增此參數：大於 1.0 即可有效懲罰重複生成的 Token (建議值 1.05 ~ 1.2)
                no_repeat_ngram_size=5   # 新增此參數：禁止連續 5 個 token 的片語重複出現
            )

        # 解碼輸出
        answer = parser.tokenizer.decode(
            gen_outputs[0][gen_inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        print("\n💡 模型最終回答：")
        print(answer)
        print("=" * 60)
        ragas_questions.append(query)
        ragas_contexts.append(current_query_contexts) # 這裡加入的是 List
        ragas_answers.append(answer)

        # 修正點 C：單輪對話結束後再次清理顯存，避免碎片化 (Fragmentation) 累積
        del gen_inputs
        del gen_outputs
        gc.collect()
        torch.cuda.empty_cache()

        # === 迴圈結束 ===

    # 新增：建立 Ragas 可用的 Dataset
    print("\n[Step 6] 建立 Ragas 評估資料集...")
    data_dict = {
        "question": ragas_questions,
        "contexts": ragas_contexts,
        "answer": ragas_answers,
        "ground_truth": ragas_ground_truths
    }

    ragas_dataset = Dataset.from_dict(data_dict)

    # 1. 掛載 Google Drive (執行後會跳出授權視窗)
    drive.mount('/content/drive')

    # 2. 定義存檔路徑 (存到您的雲端硬碟根目錄)
    save_path = '/content/drive/MyDrive/ragas_temp_data.json'

    # 3. 將 data_dict 儲存為 JSON 檔案
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=4)

    print(f"✅ 資料已安全儲存至：{save_path}")

    # 檢查最終資料格式
    print(ragas_dataset)
    # 此時您的 ragas_dataset 已經可以送入 evaluate(ragas_dataset, metrics=[...]) 進行評估了！

if __name__ == "__main__":
    main()

=== 系統初始化開始 ===
[Step 1] 載入意圖解析暨生成模型 (4-bit)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

=== 系統初始化完成 ===


使用者提問: 如果我在國外發生嚴重意外需要醫療專機協助，這項專機服務有涵蓋哪些特定的國家或地區嗎？
------------------------------------------------------------
[Step 1] 查詢路由: text_qa
         解析條件: {'company_name': '國泰人壽'}
         搜尋字串: 嚴重意外醫療專機服務涵蓋國家或地區
[Step 2 & 3] 雙軌檢索與動態壓縮完成，產出 8 個父文本區段
[Step 4] 重排完成，保留 2 個高分文本
  ➤ 推薦排名 1 (分數: 0.4375) [純文本]
  ➤ 推薦排名 2 (分數: 0.2334) [純文本]
[Step 5] 複用 Qwen2.5 進行最終答案生成...


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



💡 模型最終回答：
【結論】：
【說明】：根據文獻1中的【第二條 名詞定義】，該附約所指的「服務區域」涵蓋了中國大陸（含香港、澳门）、日本、南韓等共14個國家或地區。
【資料來源】【文獻 1】：富邦人 Lifetime Insurance Co., Ltd. - 富邦悠遊保網路投保旅行平安保險海外醫療專機運送上附約 (實物給付類型) 第二條 名詞定义。

使用者提問: 如果被保險人不幸因意外事故身故，家屬（受益人）向保險公司申請理賠時，必須準備哪些文件？
------------------------------------------------------------
[Step 1] 查詢路由: text_qa
         解析條件: {'company_name': '富邦人壽'}
         搜尋字串: 意外身故 理賠 文件
[Step 2 & 3] 雙軌檢索與動態壓縮完成，產出 4 個父文本區段
[Step 4] 重排完成，保留 2 個高分文本
  ➤ 推薦排名 1 (分數: 0.8905) [純文本]
  ➤ 推薦排名 2 (分數: 0.7705) [純文本]
[Step 5] 複用 Qwen2.5 進行最終答案生成...

💡 模型最終回答：
【結論】：
受益人需準備保險金申請書、保險單或謄本、相驗屍体證明書或死亡诊断书、被保险人除戶戶口謄本、受益人的身份证明。

【說明】：
1. 保險金申請書：用于正式提交理賠申請。
2. 保險單或其摹本：確認保單信息。
3. 相驗屍体證明书或死亡诊斷書：證明被保險人身故事實。
4. 被保險人的除戶戶口摹本：確認被保險人身份。
5. 受益人的身份证明：確認受益人身份。

【資料來源】【第十五條 及 第十四條】：富邦人@dynamic_insurance_policy_2023-附約-第15條, 第14條

使用者提問: 條款中對於「意外傷害事故」的明確定義是什麼？
------------------------------------------------------------
[Step 1] 查詢路由: text_qa
         解析條件: {}
         搜尋字串: 意外傷害事故 定義
[Step 2 & 3] 雙軌檢索與動態壓縮完成，產出 9 個父文本區段
[Step 

RAGAS 評估

In [2]:
!pip uninstall -y ragas langchain langchain-core langchain-community langchain-google-genai

Found existing installation: ragas 0.1.21
Uninstalling ragas-0.1.21:
  Successfully uninstalled ragas-0.1.21
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-google-genai 4.2.2
Uninstalling langchain-google-genai-4.2.2:
  Successfully uninstalled langchain-google-genai-4.2.2


In [3]:
!pip install \
ragas==0.1.21 \
langchain==0.1.20 \
langchain-core==0.1.52 \
langchain-community==0.0.38 \
langchain-google-genai==1.0.3 \
google-generativeai==0.5.4

  Using cached ragas-0.1.21-py3-none-any.whl.metadata (5.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.2.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.16-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.15-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.14-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.13-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.1.11-py3

In [8]:
!pip install \
ragas==0.2.10 \
langchain-google-genai==2.0.8 \
google-generativeai==0.8.3

  Using cached langsmith-0.8.5-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using ca

In [4]:
import json
import time
import os

from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragas.metrics import (
    faithfulness,
    answer_correctness,
    answer_relevancy,
    context_precision
)

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from google.colab import userdata

# ==========================================
# API KEY
# ==========================================

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# ==========================================
# LLM
# ==========================================
# 1. 初始化原生的 LangChain 模型 (可以把 temperature 拿掉，交給 Ragas 控制)
base_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

# 2. 使用 Ragas Wrapper 包裝起來 (這是解決報錯的關鍵)
eval_llm = LangchainLLMWrapper(base_llm)
eval_embeddings = LangchainEmbeddingsWrapper(embeddings)
# ==========================================
# METRICS
# ==========================================

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision
]

# ==========================================
# LOAD DATA
# ==========================================

load_path = "/content/drive/MyDrive/ragas_temp_data.json"

print("載入資料中...")

with open(load_path, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

full_dataset = Dataset.from_dict(loaded_data)

total_questions = len(full_dataset)

print(f"成功載入資料，共 {total_questions} 題")
print("-" * 50)

# ==========================================
# SCORE
# ==========================================

accumulated_scores = {
    "faithfulness": 0.0,
    "answer_correctness": 0.0,
    "answer_relevancy": 0.0,
    "context_precision": 0.0
}

# ==========================================
# EVALUATE
# ==========================================

for i in range(total_questions):

    single_item = full_dataset.select([i])

    question_text = single_item["question"][0]

    print(f"\n▶ 正在評估第 {i+1}/{total_questions} 題")
    print(f"問題: {question_text}")

    try:

        result = evaluate(
            dataset=single_item,
            metrics=metrics,
            llm=eval_llm,
            embeddings=eval_embeddings,
            raise_exceptions=False
        )

        score_dict = result.to_pandas().iloc[0].to_dict()

        print(f"Faithfulness: {score_dict.get('faithfulness', 0):.4f}")
        print(f"Correctness: {score_dict.get('answer_correctness', 0):.4f}")
        print(f"Relevancy: {score_dict.get('answer_relevancy', 0):.4f}")
        print(f"Context Precision: {score_dict.get('context_precision', 0):.4f}")

        for k in accumulated_scores.keys():

            val = score_dict.get(k, 0)

            if str(val).lower() != "nan":
                accumulated_scores[k] += val

    except Exception as e:

        print(f"錯誤: {e}")

    time.sleep(20)

# ==========================================
# FINAL SCORE
# ==========================================

print("\n" + "=" * 50)
print("最終平均分數")

for metric_name, total_score in accumulated_scores.items():

    avg_score = total_score / total_questions

    print(f"{metric_name}: {avg_score:.4f}")

print("=" * 50)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

載入資料中...
成功載入資料，共 30 題
--------------------------------------------------

▶ 正在評估第 1/30 題
問題: 如果我在國外發生嚴重意外需要醫療專機協助，這項專機服務有涵蓋哪些特定的國家或地區嗎？


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Faithfulness: 0.6000
Correctness: 0.0000
Relevancy: 0.7480
Context Precision: 0.5000

▶ 正在評估第 2/30 題
問題: 如果被保險人不幸因意外事故身故，家屬（受益人）向保險公司申請理賠時，必須準備哪些文件？


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

...

Faithfulness: 0.4286
Correctness: 0.0000
Relevancy: 0.0000
Context Precision: 0.0000

▶ 正在評估第 29/30 題
問題: 搭乘的飛機在海外失蹤找不到遺體，受益人必須等民法宣告死亡多年才能申請身故金嗎？契約有特別時效嗎？


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Faithfulness: 0.4444
Correctness: 0.0000
Relevancy: 0.8209
Context Precision: 0.0000

▶ 正在評估第 30/30 題
問題: 在新安東京海上產險的海外緊急援助服務中，被保險人如果需要在事故當地安排海外喪葬事宜，其費用給付責任的上限是多少元？


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Faithfulness: 0.6250
Correctness: 0.0000
Relevancy: 0.0000
Context Precision: 0.0000

最終平均分數
faithfulness: 0.4398
answer_correctness: 0.0000
answer_relevancy: 0.6034
context_precision: 0.4500


避免跑到一半有幾個請求失敗的單獨測試版本

In [5]:


# ==========================================
# 4. 單題測試函數定義
# ==========================================
def evaluate_single_index(idx):
    """
    輸入資料集的 Index (0 到 n-1)，印出該題詳細內容並進行 Ragas 評估。
    """
    if idx < 0 or idx >= total_questions:
        print(f"❌ 錯誤：輸入的 Index ({idx}) 超出範圍！請輸入 0 到 {total_questions - 1} 之間的數字。")
        return

    # 取出單題資料
    single_item = full_dataset.select([idx])

    # 提取資料細節，方便肉眼除錯
    q_text = single_item["question"][0]
    ctx_length = len(single_item["contexts"][0])
    ans_text = single_item["answer"][0]
    gt_text = single_item["ground_truth"][0] if "ground_truth" in single_item.features else "無"

    print(f"\n▶ 正在單獨評估第 {idx + 1} 題 (Index: {idx})")
    print("=" * 60)
    print(f"❓ 【問題】:\n{q_text}\n")
    print(f"📚 【檢索條文 (Contexts)】: 找到了 {ctx_length} 段文本")
    print(f"🤖 【LLM 實際回答】:\n{ans_text}\n")
    print(f"✅ 【您的標準答案】:\n{gt_text}")
    print("=" * 60)
    print("⏳ Ragas 評估中，請稍候...\n")

    try:
        # 執行 Ragas 評估
        result = evaluate(
            dataset=single_item,
            metrics=metrics,
            llm=eval_llm,
            embeddings=eval_embeddings,
            raise_exceptions=False
        )

        score_dict = result.to_pandas().iloc[0].to_dict()

        print("📊 【最終評估分數】")
        print(f" └─ 🛡️ Faithfulness (忠實度): {score_dict.get('faithfulness', 0):.4f}")
        print(f" └─ ✅ Correctness (正確性): {score_dict.get('answer_correctness', 0):.4f}")
        print(f" └─ 🎯 Relevancy (相關性): {score_dict.get('answer_relevancy', 0):.4f}")
        print(f" └─ 🔍 Context Precision (檢索精確度): {score_dict.get('context_precision', 0):.4f}")

    except Exception as e:
        print(f"❌ 評估發生錯誤: {e}")

# ==========================================
# 5. 執行單獨測試
# ==========================================
# 💡 在這裡修改您想測試的 Index！
# 注意：Python 的 index 從 0 開始。若想測試第 14 題，請輸入 13。

target_index = 17
evaluate_single_index(target_index)


▶ 正在單獨評估第 18 題 (Index: 17)
❓ 【問題】:
被保險人遭遇意外身故，但當初填要保書時沒填受益人，理賠金會給誰？是變遺產還是有優先繼承人？

📚 【檢索條文 (Contexts)】: 找到了 2 段文本
🤖 【LLM 實際回答】:
【結論】：
被保險人遭遇意外地故，且未指定受益人時，理賠款項將成為被保險人的遺產。
【說明】：
1. 根據富邦人壽和台灣人壽的保險條款，意外失能或身故的保險金受益人通常為被保險人本身，不可指定或變更。
2. 若被保險人身故且未指定受益人在保險事故發生前，則按照民法繼承規定，被保險人的法定繼承人將成為該項保險金的受益者。
3. 如果沒有其他指定的受益人，保險金將視為被保險人的遺産，由其法定繼承人依照民法繼承順序取得。
【資料來源]】：
富邦人壽-富邦人壽悠游保網路投保旅行平安全約-第十七條  
台灣人壽-e起趣旅行平安保险合約-第十九條

✅ 【您的標準答案】:
被保險人遭遇意外身故，但當初填要保書時沒填受益人，理賠金會給誰？是變遺產還是有優先繼承人？ 如果當初沒有指定受益人，理賠金會給「被保險人之法定繼承人」。 條款明定，若身故保險金尚未給付或受益人先於被保險人身故且未另行指定時，以法定繼承人為受益人，且法定繼承人之順序及應得保險金之比例，會直接「適用民法繼承編相關規定」來分配。
⏳ Ragas 評估中，請稍候...



Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

📊 【最終評估分數】
 └─ 🛡️ Faithfulness (忠實度): 0.2500
 └─ ✅ Correctness (正確性): 0.0000
 └─ 🎯 Relevancy (相關性): 0.8850
 └─ 🔍 Context Precision (檢索精確度): 1.0000
